## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

In [2]:
!pip install agents


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 25.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for agents: filename=agents-1.4.0-py3-none-any.whl size=62745 sha256=4e266121d0668888b1481cd649c097d09a24045d1888c52963f01abfe1cd43f8
  Stored in directory: /Users/lewagon/Library/Caches/pip/wheels/e5/82/e5/2790dbbc1ad6037f1001bc436ea963e0877fff918dddc74fe2
  Created wheel for gym: filename=gym-0.26.2-py3-none-any.whl size=827727 sha256=dac26ac219c97de607b0849fcd489a290192c6b6945525e3a6608deb4d8c1d31
  Stored in directory: /Users/lewagon/Library/Caches/pip/wheels/95/51/6c/9bb05ebbe7c5cb8171dfaa3611f32622ca4658d53f31c79077
Successfully built agents gym
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [agents]━━━━ 3/5 [gym]


In [3]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


AttributeError: module 'tensorflow' has no attribute 'contrib'

In [2]:
load_dotenv(override=True)

True

## OpenAI Hosted Tools

OpenAI Agents SDK includes the following hosted tools:

The `WebSearchTool` lets an agent search the web.  
The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.

### Important note - API charge of WebSearchTool

This is costing me 2.5 cents per call for OpenAI WebSearchTool. That can add up to $2-$3 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 2.5 cents per call.

Costs are here: https://platform.openai.com/docs/pricing#web-search

In [ ]:
INSTRUCTIONS = "Tu es un conférencier en histoire de l'art qui nous commente la viste d'un musé. À partir d'une \
    liste de tableaux, tu explores le web pour ce tableau et produis un résumé concis des résultats. Le résumé \
    doit contenir 10 15 lignes  et faire moins de 300 mots. Capture les points principaux historique et \
    stylistique. Ce contenu sera utilisé pour comparer avec 3 autres oeuvre proches, il est donc \
    essentiel de capter le contexte, l'époque, les influences et l'approchede l'artististique. \
    N’inclus aucun commentaire supplémentaire autre que le résumé lui-même."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [ ]:
message =  "Voici 4 oeuvres: \
- Tia Peltz	The Tamer \
- Amedeo Modigliani	Monsieur Lepoutre 1916 \
- Edvard Munch	Self Portrait With Brushes 1904 \
- Lucian Freud	David Hockney \
    le tableau sur lequel je porte mon intéret: \
    - Guy Rose Girl In A Wickford Garden New England \
    Peux-tu me décire le contexte et le spécificité stylistique de ces  oeuvres en m'expliquant les points techniques marquants de chaque oeuvre. \
    Peux-tu me proposer dans un paragraphe dédié une analyse sur la proximité stylistique des ces artistes.\
    Peux tu me séparer tes réponses en chapitres avec des titres et me sortir le résultat en JSON"


with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

```json
{
  "oeuvres": [
    {
      "titre": "New York Harbor 1",
      "artiste": "John Henry Twachtman",
      "contexte": "John Henry Twachtman, peintre américain du XIXᵉ siècle, a été influencé par l'impressionnisme européen. Il a étudié à Munich et a été exposé aux techniques de peinture en plein air. Son œuvre 'New York Harbor 1' reflète son intérêt pour les paysages urbains et maritimes.",
      "techniques": "Twachtman utilise des coups de pinceau fluides et une palette de couleurs subtiles pour capturer l'atmosphère changeante du port de New York. Il privilégie les effets de lumière et d'ombre pour créer une ambiance réaliste et immersive."
    },
    {
      "titre": "Banks of the Seine at Jenfosse - Clear Weather",
      "artiste": "Claude Monet",
      "contexte": "Claude Monet, figure emblématique de l'impressionnisme français, a peint 'Banks of the Seine at Jenfosse - Clear Weather' en 1884. Cette œuvre illustre son intérêt pour les paysages fluviaux et les variations de lumière.",
      "techniques": "Monet emploie des coups de pinceau visibles et une palette de couleurs vives pour capturer les reflets de la lumière sur l'eau et la végétation environnante. Il privilégie la spontanéité et l'instantanéité, caractéristiques de l'impressionnisme. ([gallerix.org](https://gallerix.org/album/Claude-Monet/pic/glrx-2008487850?utm_source=openai))"
    },
    {
      "titre": "October 1",
      "artiste": "William Merritt Chase",
      "contexte": "William Merritt Chase, peintre américain actif à la fin du XIXᵉ siècle, a réalisé 'October 1' en 1894. Cette œuvre reflète son intérêt pour les scènes de plein air et les effets de lumière automnaux.",
      "techniques": "Chase utilise des coups de pinceau rapides et une palette de couleurs chaudes pour représenter la lumière douce de l'automne. Il capture l'atmosphère de la scène avec une approche impressionniste, en mettant l'accent sur la lumière et la couleur."
    },
    {
      "titre": "The Moret Bridge at Sunset 1892",
      "artiste": "Alfred Sisley",
      "contexte": "Alfred Sisley, peintre impressionniste britannique, a peint 'The Moret Bridge at Sunset 1892' en 1892. Cette œuvre illustre son intérêt pour les paysages fluviaux et les effets de lumière au crépuscule.",
      "techniques": "Sisley utilise des coups de pinceau délicats et une palette de couleurs douces pour capturer les nuances de la lumière du soir. Il met l'accent sur les reflets dans l'eau et l'atmosphère tranquille de la scène."
    },
    {
      "titre": "Girl in a Wickford Garden, New England",
      "artiste": "Guy Rose",
      "contexte": "Guy Rose, peintre impressionniste américain, a réalisé 'Girl in a Wickford Garden, New England' en 1908. Cette œuvre reflète son intérêt pour les scènes de jardin et les effets de lumière en Nouvelle-Angleterre.",
      "techniques": "Rose utilise des coups de pinceau fluides et une palette de couleurs vives pour capturer la lumière filtrant à travers la végétation. Il met l'accent sur les jeux de lumière et d'ombre, caractéristiques de l'impressionnisme."
    }
  ],
  "analyse_stylistique": {
    "proximite": "Les artistes mentionnés partagent une approche impressionniste, privilégiant la capture des effets de lumière et de couleur en plein air. Ils utilisent des coups de pinceau visibles et une palette de couleurs vives pour représenter la nature et les scènes de la vie quotidienne. Leurs œuvres reflètent une sensibilité commune aux variations atmosphériques et aux nuances de la lumière naturelle."
  }
}
``` 

### As always, take a look at the trace

https://platform.openai.com/traces

### We will now use Structured Outputs, and include a description of the fields

In [ ]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 20

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [ ]:

message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

In [ ]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("lion94.home@gmail.com")  # Change to your verified sender
    to_email = To("lionelmysearch@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [ ]:
send_email

In [ ]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)


In [ ]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
    "The final output will contain an index of the report, and a list of follow-up questions."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [ ]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### The next 2 functions write a report and email it

In [ ]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

### Showtime!

In [ ]:
query ="Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)
    print("Hooray!")


### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>